## 1. ライブラリのインポート

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, roc_auc_score
from xgboost import XGBClassifier
import optuna
from optuna.samplers import TPESampler

import warnings
warnings.filterwarnings('ignore')

# Plotting settings
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')

## 2. データ読み込み

In [ ]:
# Load original data
train = pd.read_csv('../data/train.csv')
test = pd.read_csv('../data/test.csv')

print(f"Train shape: {train.shape}")
print(f"Test shape: {test.shape}")

# Save test IDs for submission
test_ids = test['id'].values

# Separate features and target
X = train.drop(['id', 'diagnosed_diabetes'], axis=1)
y = train['diagnosed_diabetes']
X_test = test.drop('id', axis=1)

## 3. 特徴量エンジニアリング関数の定義

2_feature_engineering.ipynbで開発した特徴量をまとめて適用

In [ ]:
def preprocess_data(X_train, X_test, categorical_cols):
    """
    カテゴリカル変数のエンコーディング
    """
    X_train = X_train.copy()
    X_test = X_test.copy()
    
    # Label encoding
    for col in categorical_cols:
        le = LabelEncoder()
        X_train[col] = le.fit_transform(X_train[col])
        X_test[col] = le.transform(X_test[col])
    
    return X_train, X_test


def create_features(df):
    """
    全ての特徴量エンジニアリングを実行
    """
    df_new = df.copy()
    
    # 1. 健康リスク指標
    df_new['bmi_age_interaction'] = df['body_mass_index'] * df['age']
    df_new['bp_ratio'] = df['systolic_bp'] / (df['diastolic_bp'] + 1e-6)
    df_new['bp_mean'] = (df['systolic_bp'] + df['diastolic_bp']) / 2
    df_new['cholesterol_ratio'] = df['total_cholesterol'] / (df['hdl_cholesterol'] + 1e-6)
    df_new['triglyceride_hdl_ratio'] = df['triglycerides'] / (df['hdl_cholesterol'] + 1e-6)
    df_new['non_hdl_cholesterol'] = df['total_cholesterol'] - df['hdl_cholesterol']
    df_new['ldl_hdl_ratio'] = df['ldl_cholesterol'] / (df['hdl_cholesterol'] + 1e-6)
    
    # 2. カテゴリ化
    df_new['age_group'] = pd.cut(df['age'], bins=[0, 30, 40, 50, 60, 100], labels=[0, 1, 2, 3, 4]).astype(int)
    df_new['bmi_category'] = pd.cut(df['body_mass_index'], bins=[0, 18.5, 25, 30, 100], labels=[0, 1, 2, 3]).astype(int)
    df_new['bp_category'] = 0
    df_new.loc[(df['systolic_bp'] >= 120) | (df['diastolic_bp'] >= 80), 'bp_category'] = 1
    df_new.loc[(df['systolic_bp'] >= 130) | (df['diastolic_bp'] >= 85), 'bp_category'] = 2
    
    # 3. インタラクション特徴量
    df_new['gender_smoking'] = df['gender'] * df['smoking_status']
    df_new['age_family_history'] = df['age'] * df['family_history_diabetes']
    df_new['bmi_activity'] = df['body_mass_index'] * df['physical_activity_hours_per_week']
    df_new['age_alcohol'] = df['age'] * df['alcohol_consumption_per_week']
    
    # 4. 多項式特徴量
    key_features = ['age', 'body_mass_index', 'systolic_bp', 'total_cholesterol']
    for feat in key_features:
        df_new[f'{feat}_squared'] = df[feat] ** 2
    
    return df_new

## 4. データ前処理と特徴量作成の実行

In [ ]:
# Identify categorical columns
categorical_cols = X.select_dtypes(include='object').columns.tolist()
print(f"Categorical columns: {categorical_cols}")

# Preprocessing
X_processed, X_test_processed = preprocess_data(X, X_test, categorical_cols)

# Feature engineering
X_final = create_features(X_processed)
X_test_final = create_features(X_test_processed)

print(f"\nOriginal features: {X.shape[1]}")
print(f"Final features: {X_final.shape[1]}")
print(f"Added features: {X_final.shape[1] - X.shape[1]}")

## 5. ベストモデルの定義とハイパーパラメータ最適化

3_modeling.ipynbで見つけた最良のアプローチを使用

In [ ]:
def objective(trial):
    """
    Optuna objective function for XGBoost hyperparameter tuning
    """
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 100, 500),
        'max_depth': trial.suggest_int('max_depth', 3, 12),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
        'gamma': trial.suggest_float('gamma', 0, 0.5),
        'reg_alpha': trial.suggest_float('reg_alpha', 0, 1.0),
        'reg_lambda': trial.suggest_float('reg_lambda', 0, 1.0),
        'random_state': 42,
        'eval_metric': 'logloss'
    }
    
    model = XGBClassifier(**params)
    
    # 5-fold cross-validation
    cv_scores = cross_val_score(model, X_final, y, cv=5, scoring='accuracy', n_jobs=-1)
    
    return cv_scores.mean()

In [ ]:
# Run Optuna optimization
study = optuna.create_study(direction='maximize', sampler=TPESampler(seed=42))
study.optimize(objective, n_trials=100, show_progress_bar=True)

print("\n" + "="*60)
print("Optuna Optimization Results")
print("="*60)
print(f"Best CV Score: {study.best_value:.4f}")
print(f"\nBest Parameters:")
for key, value in study.best_params.items():
    print(f"  {key}: {value}")

## 6. 最終モデルの訓練と評価

In [ ]:
# Train final model with best parameters
best_params = study.best_params.copy()
best_params['random_state'] = 42
best_params['eval_metric'] = 'logloss'

final_model = XGBClassifier(**best_params)

# Train/validation split for evaluation
X_train, X_val, y_train, y_val = train_test_split(X_final, y, test_size=0.2, random_state=42, stratify=y)

# Train model
final_model.fit(X_train, y_train)

# Predictions
y_pred = final_model.predict(X_val)
y_pred_proba = final_model.predict_proba(X_val)[:, 1]

# Evaluation
print("\n" + "="*60)
print("Final Model Evaluation")
print("="*60)
print(f"Accuracy: {accuracy_score(y_val, y_pred):.4f}")
print(f"AUC-ROC: {roc_auc_score(y_val, y_pred_proba):.4f}")
print(f"\nClassification Report:\n")
print(classification_report(y_val, y_pred))

## 7. Confusion Matrixの可視化

In [ ]:
# Confusion Matrix
cm = confusion_matrix(y_val, y_pred)

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title('Confusion Matrix')
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.show()

## 8. 特徴量重要度の可視化

In [ ]:
# Feature importance
feature_importance = pd.DataFrame({
    'feature': X_final.columns,
    'importance': final_model.feature_importances_
}).sort_values('importance', ascending=False)

print("Top 20 Most Important Features:")
print(feature_importance.head(20))

# Plot
plt.figure(figsize=(10, 8))
plt.barh(range(20), feature_importance.head(20)['importance'])
plt.yticks(range(20), feature_importance.head(20)['feature'])
plt.xlabel('Importance')
plt.title('Top 20 Feature Importance')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

## 9. 全データで再訓練とテストデータ予測

In [ ]:
# Train on full training data
print("Training final model on full training data...")
final_model.fit(X_final, y)

# Predict on test data
test_predictions = final_model.predict(X_test_final)

print(f"Test predictions shape: {test_predictions.shape}")
print(f"Predicted class distribution:")
print(pd.Series(test_predictions).value_counts(normalize=True))

## 10. Submission作成

In [ ]:
# Create submission file
submission = pd.DataFrame({
    'id': test_ids,
    'diagnosed_diabetes': test_predictions
})

submission.to_csv('submission_final.csv', index=False)

print("\n" + "="*60)
print("Submission file created: submission_final.csv")
print("="*60)
print(submission.head(10))
print(f"\nSubmission shape: {submission.shape}")

## 11. クロスバリデーション最終スコア

In [ ]:
# Final cross-validation score with best model
cv_scores = cross_val_score(final_model, X_final, y, cv=5, scoring='accuracy', n_jobs=-1)

print("\n" + "="*60)
print("Final Cross-Validation Results (5-Fold)")
print("="*60)
print(f"CV Scores: {cv_scores}")
print(f"Mean CV Score: {cv_scores.mean():.4f}")
print(f"Std CV Score: {cv_scores.std():.4f}")

## まとめ

### パイプラインの流れ

1. **データ読み込み**: train.csv, test.csv
2. **前処理**: カテゴリカル変数のラベルエンコーディング
3. **特徴量エンジニアリング**: 健康リスク指標、カテゴリ化、インタラクション、多項式
4. **ハイパーパラメータ最適化**: Optuna（100 trials）
5. **モデル訓練**: XGBoost with best parameters
6. **評価**: Accuracy, AUC-ROC, Classification Report
7. **予測**: テストデータに対する予測
8. **Submission**: submission_final.csv

### 結果の改善点（前回と比較）

- より多くの特徴量を使用（元の26個 → 48個以上）
- より広範なハイパーパラメータ探索（50 trials → 100 trials）
- 正則化パラメータ（reg_alpha, reg_lambda）を追加

### 今後の改善案

- アンサンブル学習（XGBoost + LightGBM + CatBoost）
- Stacking/Blending
- より高度な特徴量選択（RFE, Boruta）
- データ拡張（SMOTE等でバランス調整）